In [ ]:
def sign(x):
    # Función auxiliar equivalente a Math.sign de JS"""
    if x > 0: return 1
    if x < 0: return -1
    return 0

def is_valid_move(piece_type, r1, c1, r2, c2, board, terrain):

  #  Valida si un movimiento es legal según las reglas de Cotadrez.
  #    Args:
  #      piece_type (str): El tipo de la unidad que se mueve.
  #      r1, c1 (int): Coordenadas de origen.
  #      r2, c2 (int): Coordenadas de destino.
  #      board (list): Matriz 10x10 que contiene None o dicts {'type': str, 'army': str}.
  #      terrain (list): Matriz 10x10 con strings (ej: 'water').

  #  Returns:
  #      bool: True si el movimiento es válido, False si no.

    # Diferencias absolutas (equivalente a Math.abs)
    dr = abs(r2 - r1)
    dc = abs(c2 - c1)

    # Obtenemos las piezas en origen y destino (si existen)
    my_piece = board[r1][c1]
    target = board[r2][c2]

    # --- 0. REGLA ESPECIAL: CAPTURA DE ELEFANTES ---
    if my_piece and target and target['type'] == 'elefante' and target['army'] != my_piece['army']:
        if piece_type == 'dragon':
            pass # El dragón captura solo sin problemas
        else:
            total_threats = count_total_threats(r2, c2, my_piece['army'], board)
            if total_threats < 2: return False

    # --- 1. REY ---
    if piece_type == 'rey':
        # La validación de Jaque se suele hacer fuera de esta función básica de geometría
        return (dr <= 1 and dc <= 1) and (dr + dc > 0)

    # --- 2. LANCEROS / CHUSMA ---
    if piece_type == 'lanceros' or piece_type == 'chusma':
        return (dr + dc == 1)

    # --- 3. DRAGÓN ---
    if piece_type == 'dragon':
        # Debe ser movimiento puro (Ortogonal o Diagonal)
        if not ((dr == dc) or (r1 == r2 or c1 == c2)):
            return False

        sr = sign(r2 - r1)
        sc = sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc

        while cr != r2 or cc != c2:
            obs = board[cr][cc]
            # Solo bloqueado por montañas ('montana')
            if obs and obs['type'] != 'montana':
                return False
            cr += sr
            cc += sc
        return True

    # --- 4. ARQUEROS (REVISADO) ---
    if piece_type == 'arqueros':
        # Movimiento Diagonal Exclusivo
        if dr != dc: return False

        # Distancia 1: Siempre posible
        if dr == 1: return True

        # Distancia 2: Habilidad "Largo Alcance"
        if dr == 2:
            # CONDICIÓN 1: Solo sirve para CAPTURAR (destino debe estar ocupado)
            if target is None: return False

            # CONDICIÓN 2: Salto limpio (casilla intermedia vacía y sin agua)
            mid_r = (r1 + r2) // 2
            mid_c = (c1 + c2) // 2

            # Verificamos bloqueo físico (pieza es None) y terreno (no es 'water')
            path_clear = (board[mid_r][mid_c] is None and terrain[mid_r][mid_c] != 'water')
            return path_clear

        return False

    # --- 5. ELEFANTE ---
    if piece_type == 'elefante':
        # Solo ortogonal
        if r1 != r2 and c1 != c2: return False

        sr = sign(r2 - r1)
        sc = sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc

        while cr != r2 or cc != c2:
            # Bloqueo TOTAL: unidades o agua detienen al elefante
            if board[cr][cc] is not None or terrain[cr][cc] == 'water':
                return False
            cr += sr
            cc += sc
        return True

    # --- 6. CABALLERÍA LIGERA (Ruta Dual) ---
    if piece_type == 'c_ligera':
        # Validar geometría en L (3x1 o 1x3)
        if not ((dr == 3 and dc == 1) or (dr == 1 and dc == 3)):
            return False

        # Función interna auxiliar para checkBlock
        def check_block(r, c):
            if r < 0 or r > 9 or c < 0 or c > 9: return True
            if terrain[r][c] == 'water': return True
            o = board[r][c]
            # Bloqueado por montañas o fortalezas
            return (o and (o['type'] == 'montana' or o['type'] == 'fortaleza'))

        # RUTA A: Salto largo (2) luego corto (1)
        path_a_valid = False
        diags2 = [(2, 2), (2, -2), (-2, 2), (-2, -2)]
        for kr, kc in diags2:
            # Buscamos la "rodilla"
            if abs(r2 - (r1 + kr)) == 1 and abs(c2 - (c1 + kc)) == 1:
                knee_r = r1 + kr
                knee_c = c1 + kc
                mid_r = (r1 + knee_r) // 2
                mid_c = (c1 + knee_c) // 2

                if not check_block(mid_r, mid_c) and not check_block(knee_r, knee_c):
                    path_a_valid = True
                break

        # RUTA B: Salto corto (1) luego largo (2)
        path_b_valid = False
        diags1 = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
        for kr, kc in diags1:
            if abs(r2 - (r1 + kr)) == 2 and abs(c2 - (c1 + kc)) == 2:
                knee_r = r1 + kr
                knee_c = c1 + kc
                mid_r = (knee_r + r2) // 2
                mid_c = (knee_c + c2) // 2

                if not check_block(knee_r, knee_c) and not check_block(mid_r, mid_c):
                    path_b_valid = True
                break

        return (path_a_valid or path_b_valid)

    # --- 7. CABALLERÍA PESADA ---
    if piece_type == 'c_pesada':
        # Validar geometría en L (2x1 o 1x2)
        if not ((dr == 2 and dc == 1) or (dr == 1 and dc == 2)):
            return False

        my_army = my_piece['army'] if my_piece else None

        def is_blocked(r, c):
            if r < 0 or r > 9 or c < 0 or c > 9: return True
            if terrain[r][c] == 'water': return True
            o = board[r][c]
            if o:
                if o['type'] == 'montana' or o['type'] == 'fortaleza': return True
                if o['army'] != my_army: return True # Enemigo bloquea
            return False

        sr = sign(r2 - r1)
        sc = sign(c2 - c1)

        # Lógica de ruta directa
        if dr == 2:
            return not (is_blocked(r1 + sr, c1) or is_blocked(r2, c1)) or \
                   not (is_blocked(r1, c2) or is_blocked(r1 + sr, c2))
        else:
            return not (is_blocked(r1, c1 + sc) or is_blocked(r1, c2)) or \
                   not (is_blocked(r2, c1) or is_blocked(r2, c1 + sc))

    # --- 8. TRABUQUETE ---
    if piece_type == 'trabuquete':
        # A. MOVIMIENTO (Destino vacío): Solo Diagonal 1 casilla
        if target is None:
            # El agua se valida en el handler principal, aquí solo geometría
            return (dr == 1 and dc == 1)

        # B. DISPARO (Destino ocupado): Solo Ortogonal
        if r1 != r2 and c1 != c2: return False

        sr = sign(r2 - r1)
        sc = sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc

        while cr != r2 or cc != c2:
            o = board[cr][cc]
            # Dispara sobre unidades y agua. Solo bloquean Muros (Montaña/Fortaleza).
            if o and (o['type'] == 'montana' or o['type'] == 'fortaleza'):
                return False
            cr += sr
            cc += sc
        return True

    # --- 9. ESCORPIÓN ---
    if piece_type == 'escorpion':
        # A. MOVIMIENTO (Destino vacío): Solo Ortogonal 1 casilla
        if target is None:
            return (dr + dc == 1)

        # B. DISPARO (Destino ocupado): Solo Diagonal
        if dr != dc: return False

        sr = sign(r2 - r1)
        sc = sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc

        enemies_in_path = 0
        last_enemy_pos = None

        while cr != r2 or cc != c2:
            o = board[cr][cc]

            # Bloqueos físicos
            if o and (o['type'] == 'montana' or o['type'] == 'fortaleza'):
                return False

            # Unidades
            if o:
                if o['army'] != my_piece['army']:
                    enemies_in_path += 1
                    last_enemy_pos = {'r': cr, 'c': cc}
                else:
                    # BLOQUEO: No puede atravesar unidades amigas
                    return False

            # Agua se ignora
            cr += sr
            cc += sc

        # Lógica de Perforación
        if enemies_in_path == 0: return True # Tiro limpio

        if enemies_in_path == 1:
            # Distancia entre la víctima intermedia y el objetivo final
            dist = abs(r2 - last_enemy_pos['r']) # Como es diagonal, dr == dc
            # Permite hueco de 0, 1 o 2 casillas
            if dist >= 1 and dist <= 3:
                return True

        return False

    return False


    def count_total_threats(target_r, target_c, army, board, terrain):

    # Cuenta cuántas unidades de un ejército específico (army) pueden atacar
    # una casilla objetivo (target_r, target_c).

    # Args:
    #    target_r, target_c (int): Coordenadas de la casilla amenazada.
    #    army (str): El ejército atacante ('rojo' o 'negro').
    #    board (list): La matriz del tablero.
    #    terrain (list): La matriz del terreno.

      threats = 0

    # Recorremos todo el tablero (10x10)
      for tr in range(10):
        for tc in range(10):
            piece = board[tr][tc]

            # Si hay pieza y es del ejército que estamos comprobando
            if piece and piece['army'] == army:

                # Verificamos si esta pieza puede atacar la casilla objetivo.
                # IMPORTANTE: Debes tener portada también la función 'can_unit_attack'
                if can_unit_attack(piece['type'], tr, tc, target_r, target_c, army, board, terrain):
                    threats += 1

    return threats


def can_unit_attack(piece_type, r1, c1, r2, c2, army, board, terrain, ignore_pos=None):
  #  Determina si una unidad en (r1, c1) puede atacar a (r2, c2).
  #  Soporta lógica de 'ignore_pos' para simulaciones (evitar autobloqueo).

    dr = abs(r2 - r1)
    dc = abs(c2 - c1)

    # --- 1. UNIDADES SIMPLES (Rey, Lanceros, Chusma) ---
    if piece_type == 'rey':
        return dr <= 1 and dc <= 1

    if piece_type == 'lanceros' or piece_type == 'chusma':
        return (dr + dc == 1)

    # --- 2. ARQUEROS ---
    if piece_type == 'arqueros':
        if dr != dc: return False # Diagonal estricta
        if dr == 1: return True   # Distancia 1 siempre ataca

        if dr == 2:
            # Calculamos punto medio (división entera)
            mid_r = (r1 + r2) // 2
            mid_c = (c1 + c2) // 2

            # Verificamos si está vacío O si es la posición ignorada (la propia unidad moviéndose)
            is_mid_empty = (board[mid_r][mid_c] is None) or \
                           (ignore_pos and mid_r == ignore_pos['r'] and mid_c == ignore_pos['c'])

            # El agua bloquea el ataque de salto (según tu regla)
            return is_mid_empty and terrain[mid_r][mid_c] != 'water'

        return False

    # --- 3. CABALLERÍA LIGERA (Ruta Dual para Amenazas) ---
    if piece_type == 'c_ligera':
        # Validar L (3x1 o 1x3)
        if not ((dr == 3 and dc == 1) or (dr == 1 and dc == 3)):
            return False

        def check_block(r, c):
            if r < 0 or r > 9 or c < 0 or c > 9: return True
            # Si es la posición a ignorar, NO bloquea
            if ignore_pos and r == ignore_pos['r'] and c == ignore_pos['c']: return False
            if terrain[r][c] == 'water': return True
            o = board[r][c]
            # Solo bloqueado por Montañas o Fortalezas
            return (o and (o['type'] == 'montana' or o['type'] == 'fortaleza'))

        # RUTA A (2+1)
        path_a = False
        diags2 = [(2, 2), (2, -2), (-2, 2), (-2, -2)]
        for kr, kc in diags2:
            if abs(r2 - (r1 + kr)) == 1 and abs(c2 - (c1 + kc)) == 1:
                knee_r = r1 + kr
                knee_c = c1 + kc
                mid_r = (r1 + knee_r) // 2
                mid_c = (c1 + knee_c) // 2
                if not check_block(mid_r, mid_c) and not check_block(knee_r, knee_c):
                    path_a = True
                break

        # RUTA B (1+2)
        path_b = False
        diags1 = [(1, 1), (1, -1), (-1, 1), (-1, -1)]
        for kr, kc in diags1:
            if abs(r2 - (r1 + kr)) == 2 and abs(c2 - (c1 + kc)) == 2:
                knee_r = r1 + kr
                knee_c = c1 + kc
                mid_r = (knee_r + r2) // 2
                mid_c = (knee_c + c2) // 2
                if not check_block(knee_r, knee_c) and not check_block(mid_r, mid_c):
                    path_b = True
                break

        return (path_a or path_b)

    # --- 4. CABALLERÍA PESADA ---
    if piece_type == 'c_pesada':
        if not ((dr == 2 and dc == 1) or (dr == 1 and dc == 2)):
            return False

        def is_blocked(r, c):
            if r < 0 or r > 9 or c < 0 or c > 9: return True
            if terrain[r][c] == 'water': return True
            if ignore_pos and r == ignore_pos['r'] and c == ignore_pos['c']: return False
            o = board[r][c]
            if o:
                if o['type'] == 'montana' or o['type'] == 'fortaleza': return True
                if o['army'] != army: return True # Enemigo bloquea
            return False

        sr = sign(r2 - r1)
        sc = sign(c2 - c1)

        # Lógica directa (igual que isValidMove pero con la función is_blocked local)
        if dr == 2:
            return not (is_blocked(r1 + sr, c1) or is_blocked(r2, c1)) or \
                   not (is_blocked(r1, c2) or is_blocked(r1 + sr, c2))
        else:
            return not (is_blocked(r1, c1 + sc) or is_blocked(r1, c2)) or \
                   not (is_blocked(r2, c1) or is_blocked(r2, c1 + sc))

    # --- 5. RAYCAST (Dragón, Elefante, Armas) ---
    def check_ray(is_ortho, is_diag, can_fly, is_weapon):
        valid_ortho = (r1 == r2 or c1 == c2)
        valid_diag = (dr == dc)

        if is_ortho and is_diag:
            if not valid_ortho and not valid_diag: return False
        elif is_ortho:
            if not valid_ortho: return False
        elif is_diag:
            if not valid_diag: return False

        sr = sign(r2 - r1)
        sc = sign(c2 - c1)
        cr, cc = r1 + sr, c1 + sc

        enemies_in_path = 0
        last_enemy_pos = None

        while cr != r2 or cc != c2:
            # Gestión de ignorePos
            if ignore_pos and cr == ignore_pos['r'] and cc == ignore_pos['c']:
                cr += sr
                cc += sc
                continue

            obs = board[cr][cc]

            if can_fly:
                # Dragón solo bloqueado por montañas
                if obs and obs['type'] != 'montana': return False
            else:
                # Terrestres/Armas
                if terrain[cr][cc] == 'water' and not is_weapon: return False

                if obs:
                    if obs['type'] == 'montana' or obs['type'] == 'fortaleza': return False

                    if is_weapon and piece_type == 'escorpion':
                        if obs['army'] != army:
                            enemies_in_path += 1
                            last_enemy_pos = {'r': cr, 'c': cc}
                        else:
                            return False # Bloqueo amigo

                    elif is_weapon and piece_type == 'trabuquete':
                        pass # Ignora unidades

                    else:
                        return False # Elefante bloqueado por cualquier cosa

            cr += sr
            cc += sc

        # Lógica especial Escorpión (Perforación)
        if piece_type == 'escorpion' and enemies_in_path > 0:
            if enemies_in_path == 1:
                dist = abs(r2 - last_enemy_pos['r'])
                return (dist >= 1 and dist <= 3)
            return False

        return True

    # Despachamos a la función interna según el tipo
    if piece_type == 'dragon':     return check_ray(True, True, True, False)
    if piece_type == 'elefante':   return check_ray(True, False, False, False)
    if piece_type == 'trabuquete': return check_ray(True, False, False, True)
    if piece_type == 'escorpion':  return check_ray(False, True, False, True)

    return False



    def get_attackers(target_r, target_c, victim_army, board, terrain):

    # Devuelve una lista de enemigos que pueden atacar la casilla (target_r, target_c).

      attackers = []
    # El enemigo es el color contrario
      enemy_army = 'negro' if victim_army == 'rojo' else 'rojo'

      for r in range(10):
        for c in range(10):
            p = board[r][c]
            # Si hay pieza y es enemiga
            if p and p['army'] == enemy_army:
                # Usamos la función que ya portaste 'can_unit_attack'
                if can_unit_attack(p['type'], r, c, target_r, target_c, enemy_army, board, terrain):
                    attackers.append({'r': r, 'c': c, 'type': p['type']})

    return attackers



    def is_simulated_move_safe(piece_type, r1, c1, r2, c2, army, board, terrain):

    # Simula un movimiento para ver si deja al Rey propio en Jaque.
    # Devuelve True si es seguro, False si es un movimiento suicida.

    # 1. Encontrar al Rey propio
      current_king_r, current_king_c = None, None

    # Optimización: Si la pieza que movemos ES el Rey, ya sabemos dónde estará (en destino r2, c2)
      if piece_type == 'rey':
        current_king_r, current_king_c = r1, c1
      else:
        # Si no, hay que buscarlo en el tablero
        found = False
        for r in range(10):
            for c in range(10):
                p = board[r][c]
                if p and p['type'] == 'rey' and p['army'] == army:
                    current_king_r, current_king_c = r, c
                    found = True
                    break
            if found: break

    # Si por algún error de datos no hay rey, asumimos seguro para no romper el juego
      if current_king_r is None: return True

    # 2. Guardar estado original para REVERTIR
      original_source = board[r1][c1]
      original_target = board[r2][c2] # Puede ser None o una pieza enemiga (captura)

    # Detectar caso especial: RELEVO de Lancero (Swap)
    # Es relevo si soy Lancero y el destino tiene un aliado
      is_relay = (piece_type == 'lanceros' and original_target and original_target['army'] == army)

      try:
        # 3. APLICAR MOVIMIENTO SIMULADO
        if is_relay:
            # Intercambio: Aliado va al origen, Lancero al destino
            board[r1][c1] = original_target
            board[r2][c2] = original_source
        else:
            # Estándar: Origen vacío, Destino ocupado
            board[r1][c1] = None
            board[r2][c2] = original_source

        # 4. CALCULAR DÓNDE QUEDA EL REY TRAS LA SIMULACIÓN
        check_r, check_c = current_king_r, current_king_c

        if piece_type == 'rey':
            # Si moví el rey, ahora está en el destino
            check_r, check_c = r2, c2
        elif is_relay and original_target['type'] == 'rey':
            # Si soy Lancero e hice relevo con el Rey, el Rey acaba en mi origen
            check_r, check_c = r1, c1

        # 5. VERIFICAR SI HAY AMENAZAS EN ESE FUTURO
        # Llamamos a get_attackers con el tablero modificado
        threats = get_attackers(check_r, check_c, army, board, terrain)

        # Si la lista de amenazas está vacía, es seguro (True). Si hay amenazas, es suicidio (False).
        return (len(threats) == 0)

      except Exception as e:
        print(f"Error en simulación: {e}")
        return True # Fallback seguro

      finally:
        # 6. REVERTIR SIEMPRE EL TABLERO (Vital)
        board[r1][c1] = original_source
        board[r2][c2] = original_target


    def apply_move(r1, c1, r2, c2, board, terrain, turn_color, dungeons):
      """
    Intenta ejecutar un movimiento en el tablero.

    Args:
        r1, c1: Coordenadas Origen.
        r2, c2: Coordenadas Destino.
        board: Estado actual del tablero.
        terrain: Mapa de terreno.
        turn_color: 'rojo' o 'negro'.
        dungeons: Diccionario {'rojo': [], 'negro': []} para guardar prisioneros.

    Returns:
        (bool, str): (Exito/Fallo, Mensaje o Estado resultante)
      """

    # 1. VALIDACIÓN BÁSICA DE ORIGEN
    piece = board[r1][c1]
    if piece is None:
        return False, "ERROR: La casilla de origen está vacía."

    if piece['army'] != turn_color:
        return False, "ERROR: No es tu turno o la pieza no es tuya."

    # 2. VALIDACIÓN DE REGLAS DE MOVIMIENTO (Nuestras funciones portadas)
    # Primero chequeamos la geometría y física
    if not is_valid_move(piece['type'], r1, c1, r2, c2, board, terrain):
        return False, "ERROR: Movimiento inválido (Geometría u obstáculos)."

    # Segundo chequeamos que no sea un suicidio (Jaque)
    if not is_simulated_move_safe(piece['type'], r1, c1, r2, c2, turn_color, board, terrain):
        return False, "ERROR: Movimiento ilegal (Deja al Rey en Jaque)."

    # 3. IDENTIFICAR EL OBJETIVO EN DESTINO
    target = board[r2][c2]

    # --- EJECUCIÓN DEL MOVIMIENTO ---

    # CASO A: RELEVO DE LANCEROS (Swap)
    # Si soy lancero y voy a una casilla aliada -> Intercambio
    if piece['type'] == 'lanceros' and target and target['army'] == turn_color:
        board[r1][c1] = target  # El aliado viene a mi sitio
        board[r2][c2] = piece   # Yo voy al suyo
        return True, "RELAY_EXECUTED"

    # CASO B: CASILLA OCUPADA POR ALIADO (No permitido si no es Lancero)
    if target and target['army'] == turn_color:
        return False, "ERROR: La casilla destino está ocupada por un aliado."

    # CASO C: CAPTURA (O DISPARO SI FUESE TRABUQUETE)
    # Nota: En tu JS, Trabuquete/Escorpión disparan sin moverse si hay target.
    # Aquí asumimos lógica de movimiento estándar o captura cuerpo a cuerpo.
    # Si quieres implementar el disparo a distancia, habría que añadir un 'if' aquí
    # para no mover la pieza origen si es artillery.

    outcome = "MOVE_EXECUTED"

    if target:
        # Es un enemigo -> Captura
        if target['type'] == 'rey':
            # ¡Jaque Mate! (O captura de Rey)
            board[r2][c2] = piece
            board[r1][c1] = None
            return True, "GAME_OVER"

        # Enviar a la mazmorra del atacante (turn_color)
        # dungeons['rojo'] guarda los prisioneros que tiene el rojo (piezas negras)
        dungeons[turn_color].append(target)
        outcome = "CAPTURE_EXECUTED"


    # Lógica específica para Artillería (Disparo a distancia)
    if piece['type'] in ['trabuquete', 'escorpion'] and target is not None:
        # Si hay objetivo, es un DISPARO, no un movimiento.
        # Eliminamos al target (ya lo guardamos en dungeon arriba)
        board[r2][c2] = None
        # PERO NO MOVEMOS LA PIEZA DE ORIGEN
        # board[r1][c1] sigue siendo 'piece'
        return True, "SHOT_FIRED"

    # EJECUTAR EL MOVIMIENTO FÍSICO
    board[r2][c2] = piece
    board[r1][c1] = None

    return True, outcome



class CotadrezGame:
    def __init__(self):
        # 1. Inicializar Tablero Vacío (10x10)
        # Cada celda será None o {'type': '...', 'army': '...'}
        self.board = [[None for _ in range(10)] for _ in range(10)]

        # 2. Inicializar Terreno (Agua)
        self.terrain = [[None for _ in range(10)] for _ in range(10)]
        self._init_water()

        # 3. Estado del juego
        self.turn_color = 'rojo' # Empieza el rojo (o quien digan las reglas)
        self.dungeons = {'rojo': [], 'negro': []}
        self.game_over = False
        self.history = [] # Para guardar el log de la partida

    def _init_water(self):
        """Configura el agua según tu lógica de JS (b4, d2 y espejos)"""
        # Coordenadas base del JS: [3, 1] y [1, 3]
        water_coords = [(3, 1), (1, 3)]
        for r, c in water_coords:
            # Aplicar simetría de 4 cuadrantes
            self.terrain[r][c] = 'water'
            self.terrain[r][9 - c] = 'water'
            self.terrain[9 - r][c] = 'water'
            self.terrain[9 - r][9 - c] = 'water'

    def coord_to_index(self, coord_str):
        """
        Convierte notación de tablero a índices de matriz.
        Ejemplo: "a10" -> (0, 0) | "j1" -> (9, 9)
        """
        cols = 'abcdefghij'
        try:
            c_char = coord_str[0].lower()
            r_str = coord_str[1:]

            c = cols.index(c_char)
            # En tu JS: fila 0 es arriba ("10"), fila 9 es abajo ("1")
            r = 10 - int(r_str)
            return r, c
        except:
            print(f"Error parseando coordenada: {coord_str}")
            return None

    # --- AQUÍ ES DONDE NECESITO TUS REGLAS DE NOTACIÓN ---
    def cargar_despliegue(self, notacion_despliegue):
        """
        Recibe un string largo con la configuración inicial.
        Ejemplo inventado: "N:Ra10,Fa9... / S:Re1,Fe2..."
        """
        print(f"--- Cargando Despliegue ---")
        # TODO: Implementar lógica cuando me pases las reglas
        pass

    def ejecutar_secuencia_movimientos(self, lista_movimientos):
        """
        Recibe una lista de strings: ["e2e4", "Ra1a2", ...]
        Y los ejecuta uno a uno validando legalidad.
        """
        print(f"\n--- Iniciando Secuencia de Movimientos ---")

        for i, move_str in enumerate(lista_movimientos):
            if self.game_over:
                print(f"El juego ya terminó. Se ignora: {move_str}")
                break

            # 1. Parsear el string (asumiendo formato estándar origen-destino por ahora)
            # Necesitaré saber si usas notación algebraica completa (Cf3) o coordenadas (g1f3)
            # Por ahora asumo coordenadas puras tipo "a2a3"

            # Ejemplo simple: "a2a3"
            src_str = move_str[:2] # "a2"
            dst_str = move_str[2:] # "a3" (si hay 4 chars)

            r1, c1 = self.coord_to_index(src_str)
            r2, c2 = self.coord_to_index(dst_str)

            # 2. Llamar a la función MAESTRA que creamos antes
            # (Asumimos que apply_move ya está importada o es método de la clase)
            exito, mensaje = apply_move(
                r1, c1, r2, c2,
                self.board, self.terrain,
                self.turn_color, self.dungeons
            )

            if exito:
                print(f"Turno {i+1} [{self.turn_color}]: {move_str} -> ✅ {mensaje}")
                # Cambiar turno
                self.turn_color = 'negro' if self.turn_color == 'rojo' else 'rojo'
                if mensaje == "GAME_OVER":
                    self.game_over = True
                    print(f"🏆 FIN DE PARTIDA")
            else:
                print(f"Turno {i+1} [{self.turn_color}]: {move_str} -> ❌ ILEGAL: {mensaje}")
                break

# --- FIN DE LA CLASE ---






import re

# (Asegúrate de tener PIECE_MAP definido como antes)
PIECE_MAP = {
    'A': 'arqueros', 'B': 'escorpion', 'C': 'c_ligera',
    'D': 'dragon',   'E': 'elefante',  'F': 'fortaleza',
    'H': 'chusma',   'L': 'lanceros',  'M': 'montana',
    'P': 'c_pesada', 'R': 'rey',       'T': 'trabuquete'
}

class CotadrezParser:
    def __init__(self, game_instance):
        self.game = game_instance
        self.in_siege_mode = False

    def parse_log(self, full_log_text):
        lines = full_log_text.strip().split('\n')

        for line in lines:
            line = line.strip()
            if not line: continue

            # --- FASES Y METADATOS ---
            if line.startswith("["): continue # Ignorar metadatos por ahora

            if line.startswith("rF"):
                self._procesar_fortaleza(line)
                continue

            if line == "//":
                print("\n🔄 CAMBIO DE DESPLIEGUE")
                self.game.turn_color = 'negro' if self.game.turn_color == 'rojo' else 'rojo'
                continue

            if line == ">>>":
                print("\n⚔️ INICIO DEL COMBATE")
                self.game.game_phase = 'playing'
                self.game.turn_color = 'rojo'
                continue

            # --- PROCESAMIENTO DE TOKENS ---
            # Separamos por espacios, pero mantenemos la agrupación lógica
            tokens = line.split()

            i = 0
            while i < len(tokens):
                token = tokens[i]

                # Ignorar números de turno "1."
                if re.match(r'^\d+\.$', token):
                    i += 1; continue

                # Control de Asedio
                if token == "->ASEDIO":
                    self.in_siege_mode = True
                    print("\n🚨 --- ALERTA DE ASEDIO ---")
                    i += 1; continue
                if token == "<-ASEDIO":
                    self.in_siege_mode = False
                    print("🏁 --- FIN DE ASEDIO ---\n")
                    i += 1; continue

                # Ignorar paréntesis de balance "(...)"
                if token.startswith("(") or token.endswith(")"):
                    i += 1; continue

                # --- CASO ESPECIAL: MARCHA FORZADA (&) ---
                if token == "&":
                    # El token '&' une dos movimientos del MISMO turno.
                    # Como el movimiento anterior ya cambió el turno, debemos REVERTIRLO
                    # para que el siguiente movimiento cuente como del jugador actual.
                    print("   ⚡ (Marcha Forzada: Mismo turno)")
                    self.game.switch_turn() # "Deshacer" el cambio de turno
                    i += 1
                    continue

                # --- PROCESAR MOVIMIENTO/ACCIÓN ---
                self._procesar_accion(token)

                # Gestión de Turnos:
                # Si estamos en Asedio, NO cambiamos turno (es una resolución)
                # Si es normal, cambiamos turno (si aparece un '&' luego, se corregirá arriba)
                if not self.in_siege_mode:
                    self.game.switch_turn()

                i += 1

    def _procesar_fortaleza(self, token):
        coords = re.findall(r'([a-j](?:10|[0-9]))', token)
        if len(coords) == 2:
            r1, c1 = self.game.coord_to_index(coords[0])
            r2, c2 = self.game.coord_to_index(coords[1])
            self.game.deploy_fortress_area(r1, c1, r2, c2)
        else:
            print(f"❌ Error Fortaleza: {token}")

    def _procesar_accion(self, token):
        # 1. Regex para RESCATE (mA, mE) - Sin destino
        match_rescue = re.match(r'^m([A-Z])$', token)
        if match_rescue:
            char = match_rescue.group(1)
            p_type = PIECE_MAP.get(char, '?')
            print(f"🚁 RESCATE DE MAZMORRA: {p_type.upper()}")
            self.game.rescue_piece(p_type)
            return

        # 2. Regex ESTÁNDAR COMPLEJO (Soporta L=Rd5, Lxe5, Le5, rLe5)
        # Grupos:
        # 1: Prefijo (r)
        # 2: Pieza Actor (L)
        # 3: Origen Opcional (e2)
        # 4: Acción (=, x)
        # 5: Pieza Objetivo (R) - SOLO PARA RELEVO (=)
        # 6: Destino (d5)
        # 7: Sufijo (+, #)
        pattern = r'^([r])?([A-Z])([a-j](?:10|[0-9]))?([x=])?([A-Z])?([a-j](?:10|[0-9]))([+#])?$'

        match = re.match(pattern, token)
        if not match:
            print(f"⚠️ No entiendo este token: {token}")
            return

        prefix, char_actor, origin_str, action, char_target, dest_str, suffix = match.groups()

        actor_type = PIECE_MAP.get(char_actor, 'unknown')
        dest_r, dest_c = self.game.coord_to_index(dest_str)

        # A) DESPLIEGUE (Prefijo 'r' + Destino)
        if prefix == 'r' and not action:
            print(f"🆕 DESPLIEGUE: {actor_type} en {dest_str}")
            self.game.deploy_unit(actor_type, dest_r, dest_c)
            return

        # B) MOVIMIENTOS / COMBATE
        origin_r, origin_c = None, None

        # Resolver Origen
        if origin_str:
            origin_r, origin_c = self.game.coord_to_index(origin_str)
        else:
            # Buscar candidato
            # Nota: Si es relevo (=), buscamos quién puede ir ahí.
            is_capture = (action == 'x')
            candidates = self.game.find_piece_candidates(actor_type, dest_r, dest_c, is_capture)

            if len(candidates) == 1:
                origin_r, origin_c = candidates[0]['r'], candidates[0]['c']
            elif len(candidates) == 0:
                print(f"❌ ILEGAL: No hay {actor_type} que pueda ir a {dest_str}")
                return
            else:
                print(f"❌ AMBIGUO: Varios {actor_type} pueden ir a {dest_str}. Falta origen.")
                return

        # Ejecutar Lógica
        if action == '=':
            # INTERCAMBIO / RELEVO (L=Rd5)
            # Verificamos que en destino esté la pieza que dice la notación (char_target)
            target_piece = self.game.board[dest_r][dest_c]
            expected_target_type = PIECE_MAP.get(char_target, '?')

            if target_piece and target_piece['type'] == expected_target_type:
                print(f"🔄 RELEVO: {actor_type} cambia posición con {expected_target_type} en {dest_str}")
                self.game.apply_move_internal(origin_r, origin_c, dest_r, dest_c) # Tu lógica interna ya maneja el swap si es aliado
            else:
                print(f"❌ Error Relevo: Se esperaba {expected_target_type} en {dest_str}, pero hay {target_piece}")

        elif action == 'x' and actor_type in ['trabuquete', 'escorpion']:
            # DISPARO
            print(f"🔥 DISPARO de {actor_type} a {dest_str}")
            self.game.execute_shot(origin_r, origin_c, dest_r, dest_c)

        else:
            # MOVIMIENTO NORMAL O CAPTURA CUERPO A CUERPO
            verb = "CAPTURA" if action == 'x' else "MUEVE"
            print(f"➡️ {verb}: {actor_type} de {origin_str or '?'} a {dest_str}")
            self.game.apply_move_internal(origin_r, origin_c, dest_r, dest_c)

# --- CLASE DE JUEGO ACTUALIZADA PARA SOPORTAR PARSER ---
class CotadrezGame:
    def __init__(self):
        self.board = [[None]*10 for _ in range(10)]
        self.terrain = [[None]*10 for _ in range(10)]
        self._init_water()
        self.turn_color = 'rojo' # J1 Rojo Sur
        # Límites de la fortaleza: {'rojo': {'min_r': X, 'max_r': Y...}, 'negro': ...}
        self.fortress_bounds = {'rojo': None, 'negro': None}

        # Reservas (simplificado)
        self.reserves = {'rojo': {}, 'negro': {}}
        for p in PIECE_MAP.values():
            self.reserves['rojo'][p] = 10
            self.reserves['negro'][p] = 10
        self.dungeons = {'rojo': [], 'negro': []}
        self.game_phase = 'setup' # setup | playing

    def _init_water(self):
        coords = [(3,1), (1,3)]
        for r, c in coords:
            self.terrain[r][c] = 'water'
            self.terrain[r][9-c] = 'water'
            self.terrain[9-r][c] = 'water'
            self.terrain[9-r][9-c] = 'water'

    def coord_to_index(self, s):
        try:
            c = 'abcdefghij'.index(s[0].lower())
            r = 10 - int(s[1:])
            return r, c
        except: return None, None

    def switch_turn(self):
        self.turn_color = 'negro' if self.turn_color == 'rojo' else 'rojo'

    # --- DESPLIEGUE DE FORTALEZA (Guarda límites) ---
    def deploy_fortress_area(self, r1, c1, r2, c2):
        army = self.turn_color
        min_r, max_r = min(r1, r2), max(r1, r2)
        min_c, max_c = min(c1, c2), max(c1, c2)

        # Validaciones de Zona y Tamaño (Igual que antes) ...
        if (max_r - min_r) != 1 or (max_c - min_c) != 1:
            print(f"❌ ILEGAL: Fortaleza debe ser 2x2.")
            return

        # Guardamos los límites para calcular el anillo luego
        self.fortress_bounds[army] = {
            'min_r': min_r, 'max_r': max_r,
            'min_c': min_c, 'max_c': max_c
        }

        # Construcción
        try:
            cells = [(min_r, min_c), (min_r, max_c), (max_r, min_c), (max_r, max_c)]
            for cr, cc in cells:
                if self.board[cr][cc]:
                    print(f"❌ Ocupado {cr},{cc}"); return
                self.board[cr][cc] = {'type': 'fortaleza', 'army': army}
            print(f"✅ Fortaleza {army} establecida en filas {min_r}-{max_r}, cols {min_c}-{max_c}.")
        except: print("❌ Error límites.")

    # --- DESPLIEGUE DE UNIDADES (Con Regla del Anillo) ---
    def deploy_unit(self, p_type, r, c):
        army = self.turn_color

        # 1. Verificar si la casilla está vacía
        if self.board[r][c] is not None:
            print(f"⚠️ Casilla {r},{c} ocupada.")
            return

        # 2. REGLA: EXCEPCIÓN DE MONTAÑAS
        if p_type == 'montana':
            # Las montañas pueden ir donde sea (o en su propio territorio)
            # Aquí permitimos libre colocación según tu indicación.
            self.board[r][c] = {'type': p_type, 'army': army}
            print(f"🏔️ Montaña colocada en {r},{c}")
            return

        # 3. REGLA: SOLO EN EL ANILLO DE SALIDA
        # Verificar que la fortaleza existe
        bounds = self.fortress_bounds[army]
        if not bounds:
            print(f"❌ ILEGAL: No puedes desplegar tropas sin haber construido tu Fortaleza primero.")
            return

        # Calcular si (r,c) está en el anillo adyacente
        # El anillo son las casillas donde:
        # La distancia en filas al bloque fortaleza es <= 1 Y
        # La distancia en columnas al bloque fortaleza es <= 1 Y
        # NO está DENTRO de la fortaleza.

        # Distancia a la "caja" de la fortaleza
        # Si r < min_r, dist es min_r - r. Si r > max_r, dist es r - max_r. Si está dentro, 0.
        d_row = max(0, bounds['min_r'] - r, r - bounds['max_r'])
        d_col = max(0, bounds['min_c'] - c, c - bounds['max_c'])

        # Para ser el anillo, la distancia máxima en cualquier eje debe ser exactamente 1
        # (Si ambas son 0, está dentro. Si alguna es >1, está lejos).
        in_ring = (max(d_row, d_col) == 1)

        if not in_ring:
            print(f"❌ ILEGAL: Despliegue en {r},{c} inválido.")
            print(f"   (Las tropas deben colocarse en el anillo adyacente a la Fortaleza).")
            return

        # Si pasa todas las reglas:
        self.board[r][c] = {'type': p_type, 'army': army}
        if self.reserves[army].get(p_type, 0) > 0:
             self.reserves[army][p_type] -= 1

        # Feedback visual simple
        print(f"🛡️ {p_type.upper()} desplegado en el anillo ({r},{c})")

    # ... (Resto de métodos: rescue_piece, execute_shot, find_piece_candidates, etc.) ...
    # Asegúrate de mantener apply_move_internal y los demás que ya tenías.
    def rescue_piece(self, p_type):
        self.reserves[self.turn_color][p_type] = self.reserves[self.turn_color].get(p_type, 0) + 1

    def execute_shot(self, r1, c1, r2, c2):
        target = self.board[r2][c2]
        if target:
            self.dungeons[self.turn_color].append(target)
            self.board[r2][c2] = None

    def apply_move_internal(self, r1, c1, r2, c2):
        p = self.board[r1][c1]
        target = self.board[r2][c2]
        if target: self.dungeons[self.turn_color].append(target)
        self.board[r2][c2] = p
        self.board[r1][c1] = None

    def find_piece_candidates(self, p_type, dest_r, dest_c, is_capture):
        candidates = []
        for r in range(10):
            for c in range(10):
                p = self.board[r][c]
                if p and p['type'] == p_type and p['army'] == self.turn_color:
                    # Aquí deberías llamar a is_valid_move real
                    # if is_valid_move(p_type, r, c, dest_r, dest_c, self.board, self.terrain):
                    candidates.append({'r': r, 'c': c})
        return candidates

# --- IMPORTAMOS LAS VALIDACIONES ANTERIORES ---
# (Pega aquí las funciones is_valid_move, can_unit_attack, etc. que hicimos antes)
# O asegúrate de que estén en el mismo archivo.
# Dummy para que el código corra sin las funciones grandes pegadas:
def is_valid_move(*args): return True

# --- PRUEBA DEL PARSER ---
if __name__ == "__main__":
    juego = CotadrezGame()
    parser = CotadrezParser(juego)

    log_ejemplo = """
    [Event "Test Gemini"]
    [J1 "Rojo"]
    [J2 "Negro"]

    rFg2h3
    rPi3
    rBh4
    rTg4
    rDf4
    rCf3
    rLf2
    rEf1
    rAi1
    rEi2
    rRh1
    rHg1
    rMe3
    rMf5
    rMh5
    //
    rFc8d9
    rMd6
    rMg6
    rMe7
    rRd10
    rEc10
    rPe10
    rDb9
    rBe9
    rCb8
    rTe8
    rLb3
    rHc7
    rLd7
    >>>
    Cc4
    L=Hc7

    Le9e8
    Lc2c3
    Axe5
    Txd5 ->ASEDIO
    Pe7
    mE
    rRh6
    <-ASEDIO
    """

    parser.parse_log(log_ejemplo)

✅ Fortaleza rojo establecida en filas 7-8, cols 6-7.
🆕 DESPLIEGUE: c_pesada en i3
🛡️ C_PESADA desplegado en el anillo (7,8)
🆕 DESPLIEGUE: escorpion en h4
❌ ILEGAL: No puedes desplegar tropas sin haber construido tu Fortaleza primero.
🆕 DESPLIEGUE: trabuquete en g4
🛡️ TRABUQUETE desplegado en el anillo (6,6)
🆕 DESPLIEGUE: dragon en f4
❌ ILEGAL: No puedes desplegar tropas sin haber construido tu Fortaleza primero.
🆕 DESPLIEGUE: c_ligera en f3
🛡️ C_LIGERA desplegado en el anillo (7,5)
🆕 DESPLIEGUE: lanceros en f2
❌ ILEGAL: No puedes desplegar tropas sin haber construido tu Fortaleza primero.
🆕 DESPLIEGUE: elefante en f1
🛡️ ELEFANTE desplegado en el anillo (9,5)
🆕 DESPLIEGUE: arqueros en i1
❌ ILEGAL: No puedes desplegar tropas sin haber construido tu Fortaleza primero.
🆕 DESPLIEGUE: elefante en i2
🛡️ ELEFANTE desplegado en el anillo (8,8)
🆕 DESPLIEGUE: rey en h1
❌ ILEGAL: No puedes desplegar tropas sin haber construido tu Fortaleza primero.
🆕 DESPLIEGUE: chusma en g1
🛡️ CHUSMA desplegado e